# Experiment 2: PCA representation diagnostics

This notebook fits nested PCA representations, compares their second-order PCC distributions with the no-PCA representation, interprets PC1–PC5 through ortholog loadings and spatial maps, and optionally runs the full PCA clustering pilot.


In [ ]:
from pathlib import Path
import sys


def find_pipeline_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "Cross Species Pipeline", cwd.parent, cwd.parent.parent]
    for candidate in candidates:
        if (candidate / "src" / "cross_species").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate Cross Species Pipeline/src/cross_species")


PIPELINE_ROOT = find_pipeline_root()
PROJECT_ROOT = PIPELINE_ROOT.parent
SRC_ROOT = PIPELINE_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
RESULTS_ROOT = PIPELINE_ROOT / "results"
DATA_ROOT = PROJECT_ROOT / "data"
print(f"Pipeline root: {PIPELINE_ROOT}")


In [ ]:
from datetime import datetime
from types import SimpleNamespace
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cross_species import load_core_run
from cross_species.pca import (
    fit_nested_pca,
    load_nested_pca_scores,
    load_saved_pca_components,
    plot_pca_ortholog_interpretation,
    run_pca_pilot,
    run_similarity_calibration,
)


In [ ]:
from cross_species import load_core_run

CORE_RUN_DIR = None  # Set explicitly, or leave None to use the newest core_* run.
if CORE_RUN_DIR is None:
    candidates = [path for path in (RESULTS_ROOT / "core").glob("core_*") if path.is_dir()]
    if not candidates:
        raise FileNotFoundError("No core run found. Run 01_full_pipeline.ipynb first.")
    CORE_RUN_DIR = max(candidates, key=lambda path: path.stat().st_mtime)
core = load_core_run(CORE_RUN_DIR)
prof = core["prof"]
C = core["C"]
graph = core["graph"]
labels = core["labels"]
display(pd.Series(core["manifest"], name="value"))


In [ ]:
SMOOTHED_HUMAN_PATH = DATA_ROOT / "sm_adata/human_GSM9629357_CS23_E2S1_bin50_sm.h5ad"
SMOOTHED_MOUSE_PATH = DATA_ROOT / "sm_adata/mouse_E1S1_165_bin50_sm.h5ad"
sm_adata_human = ad.read_h5ad(SMOOTHED_HUMAN_PATH)
sm_adata_mouse = ad.read_h5ad(SMOOTHED_MOUSE_PATH)


## Fit one nested PCA


In [ ]:
RUN_PCA_FIT = False
PCA_FIT_DIR = RESULTS_ROOT / "pca" / datetime.now().strftime("pca_fit_%Y%m%d_%H%M%S")
if RUN_PCA_FIT:
    pca_fit = fit_nested_pca(
        prof,
        max_components=500,
        output_dir=PCA_FIT_DIR,
        random_state=0,
    )
    display(pd.Series(pca_fit["configuration"], name="value"))
else:
    print("Set RUN_PCA_FIT = True to create corrected reusable PCA artifacts.")


## Select PCA artifacts


In [ ]:
PCA_ARTIFACT_DIR = None  # Set explicitly, or use newest results/pca/pca_fit_*.
if PCA_ARTIFACT_DIR is None:
    candidates = [path for path in (RESULTS_ROOT / "pca").glob("pca_fit_*") if path.is_dir()]
    if candidates:
        PCA_ARTIFACT_DIR = max(candidates, key=lambda path: path.stat().st_mtime)
if PCA_ARTIFACT_DIR is None:
    print("No PCA fit selected. Run the preceding cell or set PCA_ARTIFACT_DIR.")
else:
    print(f"PCA artifacts: {PCA_ARTIFACT_DIR}")


## Calibrate PCA-specific PCC cutoffs


In [ ]:
RUN_SIMILARITY_CALIBRATION = False
if RUN_SIMILARITY_CALIBRATION:
    if PCA_ARTIFACT_DIR is None:
        raise ValueError("Set PCA_ARTIFACT_DIR first")
    if core["edge_sets"] is None:
        raise ValueError("Core run has no saved EdgeSets; regenerate it with the new pipeline")
    pca_scores, _, score_path = load_nested_pca_scores(
        PCA_ARTIFACT_DIR, minimum_dimensions=100
    )
    calibration_dir = RESULTS_ROOT / "pca" / datetime.now().strftime(
        "similarity_calibration_%Y%m%d_%H%M%S"
    )
    calibration = run_similarity_calibration(
        prof,
        C,
        core["edge_sets"],
        pca_scores=pca_scores,
        dimensions=(50, 100),
        output_dir=calibration_dir,
        show_plots=True,
    )
    display(calibration["distribution_summary"])
    display(calibration["fidelity"])
else:
    print("Set RUN_SIMILARITY_CALIBRATION = True after fitting/selecting PCA artifacts.")


## Interpret PC1–PC5


In [ ]:
RUN_COMPONENT_INTERPRETATION = False
if RUN_COMPONENT_INTERPRETATION:
    if PCA_ARTIFACT_DIR is None:
        raise ValueError("Set PCA_ARTIFACT_DIR first")
    components, _, component_path = load_saved_pca_components(
        PCA_ARTIFACT_DIR, minimum_pcs=5
    )
    n_orthologs = int(prof.meta["n_ortho"])
    split = int(prof.meta["width_a"])
    pairs = core["ortholog_pairs"]
    if pairs is None:
        pairs = pd.DataFrame({"class_key": [""] * n_orthologs})
    aligned_for_plot = SimpleNamespace(
        genes_a=list(prof.raw_genes[:n_orthologs]),
        genes_b=list(prof.raw_genes[split:split+n_orthologs]),
        pairs=pairs,
    )
    interpretation_dir = RESULTS_ROOT / "pca" / datetime.now().strftime(
        "component_interpretation_%Y%m%d_%H%M%S"
    )
    interpretation = plot_pca_ortholog_interpretation(
        prof,
        aligned_for_plot,
        components,
        expression_by_species={"mouse": sm_adata_mouse, "human": sm_adata_human},
        pcs=range(1, 6),
        n_bar_per_tail=15,
        n_maps_per_tail=5,
        output_dir=interpretation_dir,
        show_plots=True,
    )
    display(interpretation["spatially_plotted_orthologs"])
else:
    print("Set RUN_COMPONENT_INTERPRETATION = True to create PC1–PC5 figures.")


## Optional full PCA clustering pilot


In [ ]:
RUN_PCA_CLUSTERING_PILOT = False
BASELINE_MODULE_EVALUATION_DIR = None  # Completed corrected module_evaluation_* directory.
if RUN_PCA_CLUSTERING_PILOT:
    if BASELINE_MODULE_EVALUATION_DIR is None:
        raise ValueError("Set BASELINE_MODULE_EVALUATION_DIR to a corrected grid evaluation")
    pilot_dir = RESULTS_ROOT / "pca" / datetime.now().strftime("pca_pilot_%Y%m%d_%H%M%S")
    pilot = run_pca_pilot(
        prof,
        C,
        reference_neighbor_cache=None,
        expression_by_species={"mouse": sm_adata_mouse, "human": sm_adata_human},
        experiment_dir=BASELINE_MODULE_EVALUATION_DIR,
        baseline_name="run_049",
        dimensions=(50, 100, 200, 500),
        output_dir=pilot_dir,
        show_plots=True,
    )
    display(pilot["partition_agreement"])
else:
    print("The clustering pilot remains off until a corrected baseline evaluation is selected.")
